In [1]:
from confluent_kafka import Consumer, KafkaException
import json

In [ ]:
import json
from confluent_kafka import Consumer, KafkaException

# Konfigurasi consumer
consumer_config = {
    "bootstrap.servers": "kafka:9092",
    "group.id": "event-tracker",
    "auto.offset.reset": "earliest"
}

# consumer instance
consumer = Consumer(consumer_config)

# Subscribe ke topic
consumer.subscribe(["events_topic"])
print("Consumer is running and subscribed to events_topic")

# Struktur data untuk pemrosesan
user_counts = {}       # jumlah event per user
event_counts = {}      # jumlah event per jenis event
last_seen = {}         # timestamp terakhir per user
user_intervals = {}    # interval antar event per user
total_events = 0       # total semua event

try:
    while True:
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            raise KafkaException(msg.error())
    
        try: 
            # Decode payload JSON 
            data = json.loads(msg.value().decode("utf-8")) 
            user = data.get("user", "unknown")
            event_type = data.get("event", "unknown")
            timestamp = data.get("timestamp", 0)

            # Hitung total event
            total_events += 1

            # Hitung jumlah event per user 
            user_counts[user] = user_counts.get(user, 0) + 1 

            # Hitung jumlah event per jenis
            event_counts[event_type] = event_counts.get(event_type, 0) + 1

            # Catat timestamp terakhir dan interval rata-rata
            if user in last_seen:
                interval = timestamp - last_seen[user]
                user_intervals.setdefault(user, []).append(interval)
                avg_interval = sum(user_intervals[user]) / len(user_intervals[user])
                print(f"Avg interval for {user}: {avg_interval:.2f}s")
            last_seen[user] = timestamp

            # Deteksi user paling aktif
            most_active_user = max(user_counts, key=user_counts.get)

            # Print hasil pemrosesan
            output = (
                f"Received event from {user}, total count: {user_counts[user]} | "
                f"Event type {event_type}: {event_counts[event_type]} | "
                f"Total events: {total_events} | "
                f"Most active user: {most_active_user}"
            )
            print(output)

        except Exception as e:
            print(f"Failed to process message: {msg.value()} | Error: {e}")

except KeyboardInterrupt:
    print("\nStopping consumer")

finally:
    # Tutup koneksi consumer
    consumer.close()


Consumer is running and subscribed to events_topic
Received event from user123, total count: 1 | Event type login: 1 | Total events: 1 | Most active user: user123
Avg interval for user123: 20.01s
Received event from user123, total count: 2 | Event type login: 2 | Total events: 2 | Most active user: user123
Avg interval for user123: 12.51s
Received event from user123, total count: 3 | Event type login: 3 | Total events: 3 | Most active user: user123
Avg interval for user123: 10.01s
Received event from user123, total count: 4 | Event type login: 4 | Total events: 4 | Most active user: user123
